# SING versus GTM: paired 10D R-vine simulation study

Compare **SING degrees 2 and 3** with the paper's **GTM** across training sizes **125, 500, 2,000**, margins **Gaussian, exponential, Student's t (df 2.5)**, and **10 distinct vine/shuffle seeds**. There are 90 datasets, 180 SING runs, and 90 GTM runs. Each dataset has **20,000 independently simulated test observations** shared by the models.

No PIT or nonparanormal transform is used. Both models use training means and standard deviations; score matrices are restored to original variable labels. Reported likelihoods are joint log densities **on the original observation scale**, including the affine change-of-variables correction. Compare likelihoods within each marginal family.

GTM has three decorrelation layers with 10 basis functions on [-10, 10] and a marginal layer with 10 basis functions on [-5, 5]. Thirty seeded tuning trials select adaptive lasso and first/second difference penalties; marginal splines are unpenalized. Tuning uses an internal 80/20 split of N, including fold-specific standardization and adaptive weights. Penalties and iteration counts are selected before refitting the pilot and GTM on all N. The test set is used only for final scoring.

GTM's primary ranking score is the **mean absolute globally nuclear-normalized copula log-density Hessian**, matching the existing paper-study code. SING's primary score is its initial dense map's unthresholded mean squared Hessian, as in the single-run notebook; its final unthresholded scores are also saved. GTM has no binary threshold yet, so its precision/recall/F1/MCC are missing by design. SING retains its delta-method graph metrics. Missing/failed fits are displayed explicitly; paired differences use only successful runs on the same cached dataset.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'sing_experiments/simulation_common.py').exists())
sys.path.insert(0, str(ROOT))
from sing_experiments.simulation_common import (
    collect_results, paired_comparisons, read_json, select_cases,
)

RESULTS_DIR = ROOT / 'sing_experiments/results/sing_gtm_comparison'
RUN_LABEL = 'default'
GTM_SCORE = 'global_normalized_hessian'  # also 'squared_hessian' or 'pseudo_precision'
MODEL_ORDER = ['SING degree 2', 'SING degree 3', 'GTM']
COLORS = {'SING degree 2': '#2563eb', 'SING degree 3': '#7c3aed', 'GTM': '#dc2626'}


## Coverage and failures

The runners write one JSON result per fit and resume successful runs automatically. Re-run failed jobs after inspecting their `failure.txt` and `run.log`; they are retried rather than counted as successful. This notebook reads the JSON files directly, so concurrent jobs do not contend for a shared CSV.

In [2]:
results = collect_results(RESULTS_DIR, RUN_LABEL)
study_path = RESULTS_DIR / 'study.json'
study = read_json(study_path if study_path.exists() else ROOT / 'sing_experiments/simulation_study.json')
expected_models = [f'SING degree {degree}' for degree in study['sing_degrees']] + ['GTM']
expected = pd.DataFrame([
    dict(model=model, n_train=case.n_train, marginals=case.marginals, seed=case.seed)
    for case in select_cases(study) for model in expected_models
])
keys = ['model', 'n_train', 'marginals', 'seed']
if results.empty:
    completed = pd.DataFrame()
    coverage = expected.assign(status='not run')
    print('No fitted results yet. Run the scripts described in SIMULATION_STUDY.md, then rerun this notebook.')
else:
    if results.duplicated(keys).any():
        raise ValueError('Duplicate model/case rows; choose one RUN_LABEL.')
    gtm_mask = results.model.eq('GTM') & results.status.eq('ok')
    gtm_column = f'roc_auc_{GTM_SCORE}'
    if gtm_mask.any():
        if gtm_column not in results:
            raise ValueError(f'Saved GTM results have no {GTM_SCORE} score.')
        results.loc[gtm_mask, 'roc_auc'] = results.loc[gtm_mask, gtm_column]
    coverage = expected.merge(results[keys + ['status']], on=keys, how='left')
    coverage['status'] = coverage.status.fillna('not run')
    completed = results.loc[results.status.eq('ok')].copy()
    # Validate dataset pairing before aggregating results.
    paired_comparisons(results, 'roc_auc')
display(coverage.groupby(['model', 'status']).size().rename('runs').unstack(fill_value=0))
if not results.empty:
    failures = results.loc[results.status.isin(['failed', 'interrupted'])]
    if not failures.empty:
        display(failures.reindex(columns=keys + ['error', 'result_path']))


No fitted results yet. Run the scripts described in SIMULATION_STUDY.md, then rerun this notebook.


status         not run
model                 
GTM                 90
SING degree 2       90
SING degree 3       90

## ROC-AUC and test log likelihood

Lines show the mean across successful seeds; error bars show **one standard error across seeds**. Faint points show individual runs. Counts are shown in the summary table. These panels use all successful fits per model; the paired comparison below restricts to matched seeds.

In [3]:
if not completed.empty:
    summary = completed.groupby(['marginals', 'n_train', 'model']).agg(
        n_runs=('roc_auc', 'count'),
        roc_auc_mean=('roc_auc', 'mean'), roc_auc_sd=('roc_auc', 'std'),
        test_loglik_mean=('test_loglik_mean', 'mean'), test_loglik_sd=('test_loglik_mean', 'std'),
        train_loglik_mean=('train_loglik_mean', 'mean'),
    ).reset_index()
    display(summary)
else:
    summary = pd.DataFrame()


In [4]:
def plot_metric(metric, label, limits=None):
    if completed.empty:
        print(f'{label}: waiting for completed fits.')
        return
    families = study['marginals']
    sizes = study['train_sizes']
    fig, axes = plt.subplots(1, len(families), figsize=(5 * len(families), 4), squeeze=False,
                             constrained_layout=True)
    for ax, family in zip(axes[0], families):
        for offset, model in zip([-0.12, 0.0, 0.12], MODEL_ORDER):
            subset = completed.loc[completed.marginals.eq(family) & completed.model.eq(model)]
            if subset.empty:
                continue
            stats = subset.groupby('n_train')[metric].agg(['mean', 'std', 'count']).reindex(sizes)
            positions = np.arange(len(sizes)) + offset
            for j, size in enumerate(sizes):
                values = subset.loc[subset.n_train.eq(size), metric].to_numpy()
                ax.scatter(np.full(len(values), positions[j]), values, color=COLORS[model], alpha=0.25, s=18)
            ax.plot(positions, stats['mean'], marker='o', color=COLORS[model], label=model)
            uncertainty = stats['std'] / np.sqrt(stats['count'])
            valid = np.isfinite(uncertainty.to_numpy())
            if valid.any():
                ax.errorbar(positions[valid], stats['mean'].to_numpy()[valid],
                            yerr=uncertainty.to_numpy()[valid], fmt='none', capsize=4, color=COLORS[model])
        ax.set(xticks=np.arange(len(sizes)), xticklabels=[f'{n:,}' for n in sizes],
               xlabel='Training observations', ylabel=label, title=family.capitalize())
        if limits is not None:
            ax.set_ylim(*limits)
        if metric == 'roc_auc':
            ax.axhline(0.5, color='#94a3b8', linestyle='--', linewidth=1)
        ax.grid(axis='y', alpha=0.2)
    legend = {}
    for ax in axes[0]:
        handles, labels = ax.get_legend_handles_labels()
        legend.update(zip(labels, handles))
    if legend:
        fig.legend(list(legend.values()), list(legend), loc='outside upper center', ncol=3)
    plt.show()

plot_metric('roc_auc', 'ROC-AUC', limits=(0, 1.02))
plot_metric('test_loglik_mean', 'Test mean joint log likelihood')


ROC-AUC: waiting for completed fits.
Test mean joint log likelihood: waiting for completed fits.


## Paired comparisons

Positive differences favor GTM. `n_pairs` counts seeds with a successful GTM and the corresponding successful SING fit on the identical data. Standard errors describe variability across these paired seeds; no missing result is replaced by a favorable score.

In [5]:
paired_auc = paired_comparisons(results, 'roc_auc')
paired_loglik = paired_comparisons(results, 'test_loglik_mean')
paired = pd.concat([paired_auc, paired_loglik], ignore_index=True)
if not paired.empty:
    display(paired)
else:
    print('No complete GTM/SING pairs yet.')


No complete GTM/SING pairs yet.


## Other stored metrics and exports

Train and test likelihoods are saved as sums and means, along with per-observation log densities. Thresholded SING metrics use the final SING graph, and ranking scores use the initial dense fit. GTM ranking matrices are available in each run's `evaluation.npz`, including global normalized Hessians, squared Hessians, absolute Hessians, and pseudo-precision. GTM binary graph metrics remain undefined until a thresholding rule is chosen independently of the ground-truth labels.

In [6]:
if not completed.empty:
    display(completed.reindex(columns=keys + [
        'roc_auc', 'precision', 'recall', 'f1', 'mcc',
        'train_loglik_mean', 'test_loglik_mean', 'train_loglik_sum', 'test_loglik_sum',
        'device', 'fit_seconds', 'threshold_method',
    ]).sort_values(keys))
    results.to_csv(RESULTS_DIR / f'metrics_{RUN_LABEL}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'summary_{RUN_LABEL}.csv', index=False)
    paired.to_csv(RESULTS_DIR / f'paired_comparisons_{RUN_LABEL}.csv', index=False)
    print('Exported metric, summary, and paired-comparison CSVs to', RESULTS_DIR)
